In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import warnings
import os
from datetime import datetime
import json
import pickle
warnings.filterwarnings('ignore')

# ===================== OBJETIVOS DE RENDIMIENTO =====================
MAE_TARGET = 3.0          # MAE general objetivo
PEAK_MAE_TARGET = 4.0     # MAE en picos (>P90) objetivo
R2_TARGET = 0.87          # R² objetivo


def check_objectives(mae, peak_mae, r2):
    """Verifica si se cumplen los tres objetivos simultáneamente."""
    return (mae < MAE_TARGET) and (peak_mae < PEAK_MAE_TARGET) and (r2 > R2_TARGET)


def objective_score(mae, peak_mae, r2):
    """
    Score compuesto que mide qué tan cerca estamos de los objetivos.
    Menor es mejor. Penaliza el incumplimiento de cada objetivo.
    """
    # Normalización respecto a los objetivos
    mae_norm = mae / MAE_TARGET
    peak_norm = peak_mae / PEAK_MAE_TARGET
    r2_norm = R2_TARGET / max(r2, 1e-6)  # invertido porque mayor R² es mejor

    # Score ponderado (MAE general 40%, MAE picos 35%, R² 25%)
    return 0.40 * mae_norm + 0.35 * peak_norm + 0.25 * r2_norm


class DengueMeteoFeatureEngineeringLGBM:
    """
    Clase optimizada para predicción de casos de dengue usando SOLO
    variables meteorológicas con LightGBM.
    """

    # Lista blanca de predictores meteorológicos permitidos
    METEOROLOGICAL_PREDICTORS = [
        'fecha', 'semana_epi',
        'temp', 'temp_max', 'temp_min', 'hum_esp', 'hum_rel', 'prec', 'dias_lluvia',
        # Rezagos de temperatura
        'temp_lag_1', 'temp_lag_2', 'temp_lag_3', 'temp_lag_4', 'temp_lag_5', 'temp_lag_6',
        'temp_lag_7', 'temp_lag_8', 'temp_lag_9', 'temp_lag_10', 'temp_lag_11', 'temp_lag_12',
        'temp_max_lag_1', 'temp_max_lag_2', 'temp_max_lag_3', 'temp_max_lag_4', 'temp_max_lag_5',
        'temp_max_lag_6', 'temp_max_lag_7', 'temp_max_lag_8', 'temp_max_lag_9', 'temp_max_lag_10',
        'temp_max_lag_11', 'temp_max_lag_12',
        'temp_min_lag_1', 'temp_min_lag_2', 'temp_min_lag_3', 'temp_min_lag_4', 'temp_min_lag_5',
        'temp_min_lag_6', 'temp_min_lag_7', 'temp_min_lag_8', 'temp_min_lag_9', 'temp_min_lag_10',
        'temp_min_lag_11', 'temp_min_lag_12',
        # Rezagos de humedad
        'hum_esp_lag_1', 'hum_esp_lag_2', 'hum_esp_lag_3', 'hum_esp_lag_4', 'hum_esp_lag_5',
        'hum_esp_lag_6', 'hum_esp_lag_7', 'hum_esp_lag_8', 'hum_esp_lag_9', 'hum_esp_lag_10',
        'hum_esp_lag_11', 'hum_esp_lag_12',
        'hum_rel_lag_1', 'hum_rel_lag_2', 'hum_rel_lag_3', 'hum_rel_lag_4', 'hum_rel_lag_5',
        'hum_rel_lag_6', 'hum_rel_lag_7', 'hum_rel_lag_8', 'hum_rel_lag_9', 'hum_rel_lag_10',
        'hum_rel_lag_11', 'hum_rel_lag_12',
        # Rezagos de precipitación
        'prec_lag_1', 'prec_lag_2', 'prec_lag_3', 'prec_lag_4', 'prec_lag_5', 'prec_lag_6',
        'prec_lag_7', 'prec_lag_8', 'prec_lag_9', 'prec_lag_10', 'prec_lag_11', 'prec_lag_12',
        'dias_lluvia_lag_1', 'dias_lluvia_lag_2', 'dias_lluvia_lag_3', 'dias_lluvia_lag_4',
        'dias_lluvia_lag_5', 'dias_lluvia_lag_6', 'dias_lluvia_lag_7', 'dias_lluvia_lag_8',
        'dias_lluvia_lag_9', 'dias_lluvia_lag_10', 'dias_lluvia_lag_11', 'dias_lluvia_lag_12',
        # Rezagos oceánicos
        'soi_lag_8', 'soi_lag_9', 'soi_lag_10', 'soi_lag_11', 'soi_lag_12',
        'sst_lag_8', 'sst_lag_9', 'sst_lag_10', 'sst_lag_11', 'sst_lag_12',
    ]

    def __init__(self, n_features_to_select=25):
        self.n_features_to_select = n_features_to_select
        self.scaler = StandardScaler()
        self.rfe = None
        self.selected_features = None
        self.feature_names = None
        self.lgbm_model = None
        self.numeric_columns = None
        self.best_params = None
        self.X_augmented_columns = None
        self.best_metrics = None
        self.train_val_metrics = None
        self.feature_selection_results = None

    # ---------- FILTRADO DE PREDICTORES ----------
    def filter_meteorological_predictors(self, X):
        """Mantiene únicamente los predictores meteorológicos permitidos."""
        allowed = [c for c in X.columns if c in self.METEOROLOGICAL_PREDICTORS]
        dropped = [c for c in X.columns if c not in allowed]
        if dropped:
            print(f"⚠️ Columnas descartadas (no meteorológicas): {len(dropped)}")
            for c in dropped[:10]:
                print(f"   - {c}")
            if len(dropped) > 10:
                print(f"   ... y {len(dropped) - 10} más")
        return X[allowed].copy()

    # ---------- INGENIERÍA DE ATRIBUTOS (solo meteorológicas) ----------
    def create_interaction_features(self, X):
        interaction_data = pd.DataFrame(index=X.index)
        interactions = [
            ('temp', 'hum_rel', 'temp_hum_rel'),
            ('temp_max', 'hum_rel', 'temp_max_hum_rel'),
            ('prec', 'temp', 'prec_temp'),
            ('dias_lluvia', 'hum_rel', 'dias_lluvia_hum_rel'),
            ('soi_lag_12', 'sst_lag_12', 'soi_sst_lag12'),
            ('temp', 'soi_lag_12', 'temp_soi_lag12'),
            ('prec', 'hum_rel', 'prec_hum_rel'),
        ]
        for col1, col2, new_col in interactions:
            if col1 in X.columns and col2 in X.columns:
                interaction_data[new_col] = X[col1] * X[col2]
        return interaction_data

    def create_polynomial_features(self, X):
        important_vars = ['temp', 'hum_rel', 'prec', 'temp_max', 'temp_min']
        poly_features = pd.DataFrame(index=X.index)
        for var in important_vars:
            if var in X.columns:
                poly_features[f'{var}_squared'] = X[var] ** 2
        return poly_features

    def create_lag_aggregates(self, X):
        """Agregados de rezagos SOLO para variables meteorológicas."""
        lag_aggs = pd.DataFrame(index=X.index)
        for base_var in ['temp', 'temp_max', 'temp_min', 'hum_esp', 'hum_rel', 'prec', 'dias_lluvia']:
            lag_cols = [c for c in X.columns if c.startswith(f'{base_var}_lag_')]
            if len(lag_cols) >= 3:
                lag_data = X[lag_cols]
                lag_aggs[f'{base_var}_lag_mean'] = lag_data.mean(axis=1)
                lag_aggs[f'{base_var}_lag_std'] = lag_data.std(axis=1)
                lag_aggs[f'{base_var}_lag_max'] = lag_data.max(axis=1)
                lag_aggs[f'{base_var}_lag_min'] = lag_data.min(axis=1)
        return lag_aggs

    def create_rolling_features(self, X):
        rolling_features = pd.DataFrame(index=X.index)
        if 'semana_epi' in X.columns:
            rolling_features['week_sin'] = np.sin(2 * np.pi * X['semana_epi'] / 52)
            rolling_features['week_cos'] = np.cos(2 * np.pi * X['semana_epi'] / 52)
        # Tendencias meteorológicas (sin usar casos_dengue)
        if 'temp' in X.columns and 'temp_lag_4' in X.columns:
            rolling_features['temp_trend_4w'] = X['temp'] - X['temp_lag_4']
        if 'temp' in X.columns and 'temp_lag_12' in X.columns:
            rolling_features['temp_trend_12w'] = X['temp'] - X['temp_lag_12']
        if 'prec' in X.columns and 'prec_lag_4' in X.columns:
            rolling_features['prec_trend_4w'] = X['prec'] - X['prec_lag_4']
        if 'hum_rel' in X.columns and 'hum_rel_lag_4' in X.columns:
            rolling_features['hum_trend_4w'] = X['hum_rel'] - X['hum_rel_lag_4']
        return rolling_features

    def get_numeric_columns(self, X):
        exclude_cols = ['fecha', 'date', 'datetime', 'timestamp']
        numeric_cols = []
        for col in X.columns:
            if col.lower() in exclude_cols:
                continue
            if pd.api.types.is_numeric_dtype(X[col]):
                numeric_cols.append(col)
        return numeric_cols

    def augment_features(self, X):
        """Ingeniería de atributos sobre predictores meteorológicos."""
        X_meteo = self.filter_meteorological_predictors(X)
        numeric_cols = self.get_numeric_columns(X_meteo)
        X_numeric = X_meteo[numeric_cols]

        features_list = [X_numeric]

        f = self.create_interaction_features(X_numeric)
        if not f.empty:
            features_list.append(f)

        f = self.create_polynomial_features(X_numeric)
        if not f.empty:
            features_list.append(f)

        f = self.create_lag_aggregates(X_numeric)
        if not f.empty:
            features_list.append(f)

        f = self.create_rolling_features(X_numeric)
        if not f.empty:
            features_list.append(f)

        X_augmented = pd.concat(features_list, axis=1)
        X_augmented = X_augmented.replace([np.inf, -np.inf], np.nan)
        X_augmented = X_augmented.fillna(X_augmented.mean())

        return X_augmented, X_numeric

    def fit_transform(self, X, y=None):
        print("Iniciando ingeniería de atributos (solo meteorológicos)...")
        X_augmented, X_numeric = self.augment_features(X)

        self.numeric_columns = X_numeric.columns.tolist()
        self.X_augmented_columns = X_augmented.columns.tolist()
        self.feature_names = X_augmented.columns.tolist()

        print("Escalando características...")
        X_scaled = pd.DataFrame(
            self.scaler.fit_transform(X_augmented),
            columns=X_augmented.columns,
            index=X_augmented.index
        )
        print(f"Características después de ingeniería: {X_augmented.shape[1]}")
        return X_scaled, X_augmented

    def transform_features(self, X):
        X_augmented, _ = self.augment_features(X)
        for col in self.X_augmented_columns:
            if col not in X_augmented.columns:
                X_augmented[col] = 0
        X_augmented = X_augmented[self.X_augmented_columns]
        X_scaled = pd.DataFrame(
            self.scaler.transform(X_augmented),
            columns=X_augmented.columns,
            index=X_augmented.index
        )
        return X_scaled

    # ---------- BÚSQUEDA DE CARACTERÍSTICAS GUIADA POR OBJETIVOS ----------
    def find_optimal_features(self, X_scaled, y, feature_range=None):
        if feature_range is None:
            feature_range = range(10, min(80, X_scaled.shape[1]), 5)

        print("\n" + "="*60)
        print("BÚSQUEDA DEL NÚMERO ÓPTIMO DE CARACTERÍSTICAS (LightGBM)")
        print(f"OBJETIVOS: MAE < {MAE_TARGET} | MAE picos < {PEAK_MAE_TARGET} | R² > {R2_TARGET}")
        print("="*60)

        results = []
        best_score = float('inf')
        best_n_features = None
        best_rfe = None
        best_selected = None
        best_metrics = None
        best_objectives_met = False

        X_train_val, X_val, y_train_val, y_val = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42
        )

        for n_features in feature_range:
            print(f"\nProbando con {n_features} características...")

            rf = RandomForestRegressor(
                n_estimators=100, max_depth=8, min_samples_split=10,
                min_samples_leaf=5, random_state=42, n_jobs=-1
            )
            rfe = RFE(estimator=rf, n_features_to_select=n_features, step=3, verbose=0)

            X_train_selected = rfe.fit_transform(X_train_val, y_train_val)
            X_val_selected = rfe.transform(X_val)

            params = {
                'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.03,
                'subsample': 0.7, 'colsample_bytree': 0.7,
                'min_child_samples': 10, 'reg_alpha': 0.5, 'reg_lambda': 2.0,
                'num_leaves': 15, 'random_state': 42, 'verbose': -1
            }
            model = lgb.LGBMRegressor(**params)
            model.fit(X_train_selected, y_train_val)
            y_pred = model.predict(X_val_selected)

            mae = mean_absolute_error(y_val, y_pred)
            r2 = r2_score(y_val, y_pred)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))

            peak_threshold = np.percentile(y_val, 90)
            peak_mask = y_val > peak_threshold
            peak_mae = mean_absolute_error(y_val[peak_mask], y_pred[peak_mask]) if np.sum(peak_mask) > 0 else float('inf')

            objectives_met = check_objectives(mae, peak_mae, r2)
            score = objective_score(mae, peak_mae, r2)

            print(f"  MAE: {mae:.4f} | R²: {r2:.4f} | RMSE: {rmse:.4f}")
            print(f"  MAE picos (>P90): {peak_mae:.4f}")
            print(f"  Objetivos cumplidos: {'✅ SÍ' if objectives_met else '❌ NO'} | Score: {score:.4f}")

            results.append({
                'n_features': n_features, 'mae': mae, 'r2': r2, 'rmse': rmse,
                'peak_mae': peak_mae, 'peak_threshold': peak_threshold,
                'n_peaks': np.sum(peak_mask),
                'objectives_met': objectives_met, 'score': score
            })

            # Prioridad: primero cumplir objetivos, luego minimizar score
            if (objectives_met and not best_objectives_met) or \
               (objectives_met == best_objectives_met and score < best_score):
                best_score = score
                best_objectives_met = objectives_met
                best_n_features = n_features
                best_rfe = rfe
                best_selected = X_scaled.columns[rfe.support_].tolist()
                best_metrics = {'mae': mae, 'r2': r2, 'rmse': rmse,
                                'peak_mae': peak_mae, 'objectives_met': objectives_met,
                                'score': score}

        print("\n" + "="*60)
        print(f"✅ NÚMERO ÓPTIMO DE CARACTERÍSTICAS: {best_n_features}")
        print(f"   MAE: {best_metrics['mae']:.4f} (objetivo < {MAE_TARGET})")
        print(f"   R²: {best_metrics['r2']:.4f} (objetivo > {R2_TARGET})")
        print(f"   MAE picos: {best_metrics['peak_mae']:.4f} (objetivo < {PEAK_MAE_TARGET})")
        print(f"   Objetivos cumplidos: {'✅ SÍ' if best_metrics['objectives_met'] else '❌ NO'}")
        print("="*60)

        self.selected_features = best_selected
        self.rfe = best_rfe
        self.best_metrics = best_metrics
        self.feature_selection_results = pd.DataFrame(results)
        return best_n_features, results, best_metrics

    def transform_with_selected_features(self, X_scaled):
        return X_scaled[self.selected_features]

    # ---------- AJUSTE DE HIPERPARÁMETROS GUIADO POR OBJETIVOS ----------
    def tune_hyperparameters(self, X_train, y_train, X_test, y_test):
        print("\n" + "="*60)
        print("AJUSTE DE HIPERPARÁMETROS LIGHTGBM (evaluación en test)")
        print(f"OBJETIVOS: MAE < {MAE_TARGET} | MAE picos < {PEAK_MAE_TARGET} | R² > {R2_TARGET}")
        print("="*60)

        param_grid = {
            'n_estimators': [200, 400, 600],
            'max_depth': [3, 4, 5, -1],
            'learning_rate': [0.01, 0.02, 0.03, 0.05],
            'subsample': [0.7, 0.8, 0.9],
            'colsample_bytree': [0.7, 0.8, 0.9],
            'min_child_samples': [5, 10, 20],
            'num_leaves': [15, 31, 63],
            'reg_alpha': [0.1, 0.3, 0.5],
            'reg_lambda': [1.0, 2.0, 5.0]
        }

        import itertools, random
        param_combinations = list(itertools.product(
            param_grid['n_estimators'], param_grid['max_depth'],
            param_grid['learning_rate'], param_grid['subsample'],
            param_grid['colsample_bytree'], param_grid['min_child_samples'],
            param_grid['num_leaves'], param_grid['reg_alpha'], param_grid['reg_lambda']
        ))
        random.seed(42)
        n_combinations = min(80, len(param_combinations))
        param_combinations = random.sample(param_combinations, n_combinations)

        print(f"Probando {len(param_combinations)} combinaciones...")

        results = []
        best_score = float('inf')
        best_objectives_met = False
        best_params = None
        best_model = None

        # Umbral de picos calculado sobre el conjunto de test
        peak_threshold_test = np.percentile(y_test, 90)

        for i, combo in enumerate(param_combinations):
            params = {
                'n_estimators': combo[0], 'max_depth': combo[1],
                'learning_rate': combo[2], 'subsample': combo[3],
                'colsample_bytree': combo[4], 'min_child_samples': combo[5],
                'num_leaves': combo[6], 'reg_alpha': combo[7],
                'reg_lambda': combo[8], 'random_state': 42, 'verbose': -1
            }

            model = lgb.LGBMRegressor(**params)
            model.fit(X_train, y_train)

            y_pred_train = model.predict(X_train)
            mae_train = mean_absolute_error(y_train, y_pred_train)
            r2_train = r2_score(y_train, y_pred_train)

            y_pred_test = model.predict(X_test)
            mae_test = mean_absolute_error(y_test, y_pred_test)
            r2_test = r2_score(y_test, y_pred_test)

            # MAE en picos del conjunto de test
            peak_mask_test = y_test > peak_threshold_test
            if np.sum(peak_mask_test) > 0:
                peak_mae_test = mean_absolute_error(
                    y_test[peak_mask_test], y_pred_test[peak_mask_test]
                )
            else:
                peak_mae_test = float('inf')

            overfit_gap = mae_test - mae_train
            objectives_met = check_objectives(mae_test, peak_mae_test, r2_test)

            # Score = objetivo + penalización por brecha de sobreajuste
            score = objective_score(mae_test, peak_mae_test, r2_test) + 0.1 * abs(overfit_gap)

            results.append({
                'params': params,
                'mae_train': mae_train, 'r2_train': r2_train,
                'mae_test': mae_test, 'r2_test': r2_test,
                'peak_mae_test': peak_mae_test,
                'overfit_gap': overfit_gap,
                'objectives_met': objectives_met, 'score': score
            })

            # Prioridad: cumplir objetivos y minimizar score
            if (objectives_met and not best_objectives_met) or \
               (objectives_met == best_objectives_met and score < best_score):
                best_score = score
                best_objectives_met = objectives_met
                best_params = params
                best_model = model

            if (i + 1) % 10 == 0:
                print(f"  Progreso: {i+1}/{len(param_combinations)} | "
                      f"Mejor score: {best_score:.4f} | "
                      f"Objetivos: {'✅' if best_objectives_met else '❌'}")

        results_df = pd.DataFrame(results).sort_values('score')

        print("\n" + "="*60)
        print("TOP 5 MODELOS SEGÚN SCORE GUIADO POR OBJETIVOS")
        print("="*60)
        for rank, (_, row) in enumerate(results_df.head(5).iterrows(), 1):
            print(f"\nModelo {rank}:")
            for k, v in row['params'].items():
                print(f"  {k}: {v}")
            print(f"  MAE Test: {row['mae_test']:.4f} | R² Test: {row['r2_test']:.4f} | "
                  f"MAE Picos: {row['peak_mae_test']:.4f}")
            print(f"  Brecha sobreajuste: {row['overfit_gap']:.4f} | "
                  f"Objetivos: {'✅' if row['objectives_met'] else '❌'}")

        # Modelo final (mejor según score guiado por objetivos)
        best_row = results_df.iloc[0]
        best_params = best_row['params']
        best_model = lgb.LGBMRegressor(**best_params)
        best_model.fit(X_train, y_train)

        print("\n" + "="*60)
        print("✅ MEJORES HIPERPARÁMETROS (guiados por objetivos):")
        print("="*60)
        for k, v in best_params.items():
            print(f"  {k}: {v}")
        print(f"\nMAE Test: {best_row['mae_test']:.4f}")
        print(f"R² Test: {best_row['r2_test']:.4f}")
        print(f"MAE Picos Test: {best_row['peak_mae_test']:.4f}")
        print(f"Objetivos cumplidos: {'✅ SÍ' if best_row['objectives_met'] else '❌ NO'}")

        self.best_params = best_params
        self.lgbm_model = best_model
        self.train_val_metrics = {
            'mae_train': best_row['mae_train'], 'r2_train': best_row['r2_train'],
            'mae_test': best_row['mae_test'], 'r2_test': best_row['r2_test'],
            'peak_mae_test': best_row['peak_mae_test'],
            'overfit_gap': best_row['overfit_gap'],
            'objectives_met': best_row['objectives_met'],
            'all_results': results_df
        }
        return best_params, best_model

    # ---------- EVALUACIÓN DETALLADA ----------
    def evaluate_model_detailed(self, X_test, y_test, dataset_name="Test"):
        y_pred = self.lgbm_model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        peak_metrics = {}
        for p in [90, 95, 97.5]:
            threshold = np.percentile(y_test, p)
            peak_mask = y_test > threshold
            if np.sum(peak_mask) > 0:
                peak_mae = mean_absolute_error(y_test[peak_mask], y_pred[peak_mask])
                peak_r2 = r2_score(y_test[peak_mask], y_pred[peak_mask]) if len(y_test[peak_mask]) > 1 else np.nan
                peak_metrics[f'P{p}_MAE'] = peak_mae
                peak_metrics[f'P{p}_R2'] = peak_r2
                peak_metrics[f'P{p}_Threshold'] = threshold
                peak_metrics[f'P{p}_Count'] = int(np.sum(peak_mask))

        objectives_met = check_objectives(mae, peak_metrics.get('P90_MAE', float('inf')), r2)

        print(f"\n{'='*60}")
        print(f"MÉTRICAS DETALLADAS - {dataset_name}")
        print(f"{'='*60}")
        print(f"MAE: {mae:.4f} (objetivo < {MAE_TARGET})")
        print(f"R²: {r2:.4f} (objetivo > {R2_TARGET})")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE P90: {peak_metrics.get('P90_MAE', float('nan')):.4f} "
              f"(objetivo < {PEAK_MAE_TARGET})")
        print(f"Objetivos cumplidos: {'✅ SÍ' if objectives_met else '❌ NO'}")

        return {
            'MAE': mae, 'R2': r2, 'RMSE': rmse,
            'peak_metrics': peak_metrics,
            'objectives_met': objectives_met,
            'predictions': y_pred,
            'actual': y_test.values if hasattr(y_test, 'values') else y_test
        }


# ===================== FUNCIONES DE GRÁFICOS =====================

def plot_hyperparameter_tuning_results(results_df, save_path=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax1 = axes[0, 0]
    ax1.scatter(results_df['mae_train'], results_df['mae_test'], alpha=0.6, s=50)
    max_v = max(results_df['mae_train'].max(), results_df['mae_test'].max())
    ax1.plot([0, max_v], [0, max_v], 'r--', alpha=0.5, label='Línea ideal')
    ax1.set_xlabel('MAE Train'); ax1.set_ylabel('MAE Test')
    ax1.set_title('MAE Train vs Test'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    ax2.scatter(results_df['r2_train'], results_df['r2_test'], alpha=0.6, s=50, color='green')
    ax2.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Línea ideal')
    ax2.set_xlabel('R² Train'); ax2.set_ylabel('R² Test')
    ax2.set_title('R² Train vs Test'); ax2.legend(); ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    ax3.hist(results_df['overfit_gap'], bins=15, edgecolor='black', alpha=0.7, color='skyblue')
    ax3.axvline(results_df['overfit_gap'].mean(), color='red', linestyle='--',
                linewidth=2, label=f'Media: {results_df["overfit_gap"].mean():.4f}')
    ax3.set_xlabel('Brecha de Sobreajuste'); ax3.set_ylabel('Frecuencia')
    ax3.set_title('Distribución de Brecha de Sobreajuste')
    ax3.legend(); ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    ax4.scatter(range(len(results_df)), results_df['score'], alpha=0.6, s=50, color='purple')
    if 'objectives_met' in results_df.columns:
        met = results_df[results_df['objectives_met']]
        if len(met) > 0:
            ax4.scatter(met.index, met['score'], color='green', s=80,
                        marker='o', label='Objetivos cumplidos')
    best_idx = results_df['score'].idxmin()
    ax4.scatter(best_idx, results_df.loc[best_idx, 'score'], color='red', s=150,
                marker='*', label='Mejor modelo')
    ax4.set_xlabel('Modelo'); ax4.set_ylabel('Score (guiado por objetivos)')
    ax4.set_title('Score Guiado por Objetivos'); ax4.legend(); ax4.grid(True, alpha=0.3)

    plt.suptitle('Ajuste de Hiperparámetros - LightGBM (meteorológicos)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Gráfico guardado en: {save_path}")
    plt.show()
    return fig


def plot_feature_selection_results(results, best_metrics, save_path=None):
    df = pd.DataFrame(results)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax1 = axes[0, 0]
    ax1.plot(df['n_features'], df['mae'], 'bo-', label='MAE General', linewidth=2)
    ax1.plot(df['n_features'], df['peak_mae'], 'ro-', label='MAE Picos', linewidth=2)
    ax1.axhline(y=MAE_TARGET, color='g', linestyle=':', label=f'Objetivo MAE={MAE_TARGET}')
    ax1.axhline(y=PEAK_MAE_TARGET, color='orange', linestyle=':', label=f'Objetivo Picos={PEAK_MAE_TARGET}')
    ax1.set_xlabel('Número de Características'); ax1.set_ylabel('MAE')
    ax1.set_title('MAE vs N° Características'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    ax2.plot(df['n_features'], df['r2'], 'go-', linewidth=2)
    ax2.axhline(y=R2_TARGET, color='r', linestyle=':', label=f'Objetivo R²={R2_TARGET}')
    ax2.set_xlabel('Número de Características'); ax2.set_ylabel('R²')
    ax2.set_title('R² vs N° Características'); ax2.legend(); ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    ax3.plot(df['n_features'], df['rmse'], 'mo-', linewidth=2)
    ax3.set_xlabel('Número de Características'); ax3.set_ylabel('RMSE')
    ax3.set_title('RMSE vs N° Características'); ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    ax4.plot(df['n_features'], df['score'], 'co-', linewidth=2)
    if 'objectives_met' in df.columns:
        met = df[df['objectives_met']]
        if len(met) > 0:
            ax4.scatter(met['n_features'], met['score'], color='green', s=100,
                        marker='o', label='Objetivos cumplidos', zorder=5)
    min_idx = df['score'].idxmin()
    ax4.plot(df['n_features'][min_idx], df['score'][min_idx], 'r*', markersize=15,
             label='Mejor (objetivos)')
    ax4.set_xlabel('Número de Características'); ax4.set_ylabel('Score guiado')
    ax4.set_title('Score Guiado por Objetivos'); ax4.legend(); ax4.grid(True, alpha=0.3)

    plt.suptitle('Selección de Características - LightGBM (meteorológicos)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Gráfico guardado en: {save_path}")
    plt.show()
    return fig


def plot_peak_analysis(y_train, y_test, train_metrics, test_metrics, save_path=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax1 = axes[0, 0]
    tr_a, tr_p = train_metrics['actual'], train_metrics['predictions']
    thr = np.percentile(tr_a, 90); mask = tr_a > thr
    ax1.scatter(tr_a[~mask], tr_p[~mask], alpha=0.5, color='blue', label='Normal', s=20)
    ax1.scatter(tr_a[mask], tr_p[mask], alpha=0.8, color='red', label='Picos', s=50)
    mn, mx = min(tr_a.min(), tr_p.min())*0.9, max(tr_a.max(), tr_p.max())*1.1
    ax1.plot([mn, mx], [mn, mx], 'r--', alpha=0.8)
    ax1.set_xlabel('Real'); ax1.set_ylabel('Predicho')
    ax1.set_title(f'Train - Picos (MAE P90: {train_metrics["peak_metrics"].get("P90_MAE",0):.3f})')
    ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_xlim(mn, mx); ax1.set_ylim(mn, mx)

    ax2 = axes[0, 1]
    te_a, te_p = test_metrics['actual'], test_metrics['predictions']
    thr = np.percentile(te_a, 90); mask = te_a > thr
    ax2.scatter(te_a[~mask], te_p[~mask], alpha=0.5, color='blue', label='Normal', s=20)
    ax2.scatter(te_a[mask], te_p[mask], alpha=0.8, color='red', label='Picos', s=50)
    mn, mx = min(te_a.min(), te_p.min())*0.9, max(te_a.max(), te_p.max())*1.1
    ax2.plot([mn, mx], [mn, mx], 'r--', alpha=0.8)
    ax2.set_xlabel('Real'); ax2.set_ylabel('Predicho')
    ax2.set_title(f'Test - Picos (MAE P90: {test_metrics["peak_metrics"].get("P90_MAE",0):.3f})')
    ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_xlim(mn, mx); ax2.set_ylim(mn, mx)

    ax3 = axes[1, 0]
    idx = np.arange(len(tr_a))
    ax3.plot(idx, tr_a, 'b-', label='Real', alpha=0.7)
    ax3.plot(idx, tr_p, 'r-', label='Predicho', alpha=0.7)
    ax3.set_xlabel('Índice'); ax3.set_ylabel('Casos')
    ax3.set_title(f'Train - Serie Temporal (MAE: {train_metrics["MAE"]:.3f})')
    ax3.legend(); ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    idx = np.arange(len(te_a))
    ax4.plot(idx, te_a, 'b-', label='Real', alpha=0.7)
    ax4.plot(idx, te_p, 'r-', label='Predicho', alpha=0.7)
    ax4.set_xlabel('Índice'); ax4.set_ylabel('Casos')
    ax4.set_title(f'Test - Serie Temporal (MAE: {test_metrics["MAE"]:.3f})')
    ax4.legend(); ax4.grid(True, alpha=0.3)

    plt.suptitle('Análisis de Picos - LightGBM (meteorológicos)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Gráfico guardado en: {save_path}")
    plt.show()
    return fig


def plot_performance_comparison(train_metrics, test_metrics, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax1 = axes[0]
    metrics = ['MAE', 'R²', 'RMSE']
    train_v = [train_metrics['MAE'], train_metrics['R2'], train_metrics['RMSE']]
    test_v = [test_metrics['MAE'], test_metrics['R2'], test_metrics['RMSE']]
    x = np.arange(len(metrics)); w = 0.35
    bars1 = ax1.bar(x - w/2, train_v, w, label='Train', color='#2E86AB', alpha=0.8)
    bars2 = ax1.bar(x + w/2, test_v, w, label='Test', color='#A23B72', alpha=0.8)
    ax1.set_xticks(x); ax1.set_xticklabels(metrics)
    ax1.set_title('Métricas Train vs Test'); ax1.legend(); ax1.grid(True, alpha=0.3)
    for b in list(bars1) + list(bars2):
        h = b.get_height()
        ax1.text(b.get_x()+b.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax2 = axes[1]
    pcts = [90, 95, 97.5]
    tr = [train_metrics['peak_metrics'].get(f'P{p}_MAE', 0) for p in pcts]
    te = [test_metrics['peak_metrics'].get(f'P{p}_MAE', 0) for p in pcts]
    x = np.arange(len(pcts)); w = 0.35
    bars1 = ax2.bar(x - w/2, tr, w, label='Train', color='#2E86AB', alpha=0.8)
    bars2 = ax2.bar(x + w/2, te, w, label='Test', color='#A23B72', alpha=0.8)
    ax2.axhline(y=PEAK_MAE_TARGET, color='r', linestyle=':',
                label=f'Objetivo picos={PEAK_MAE_TARGET}')
    ax2.set_xticks(x); ax2.set_xticklabels([f'P{p}' for p in pcts])
    ax2.set_title('MAE en Picos Train vs Test'); ax2.legend(); ax2.grid(True, alpha=0.3)
    for b in list(bars1) + list(bars2):
        h = b.get_height()
        ax2.text(b.get_x()+b.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    plt.suptitle('Comparación de Desempeño - LightGBM (meteorológicos)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Gráfico guardado en: {save_path}")
    plt.show()
    return fig


# ===================== GUARDADO DE RESULTADOS =====================

def save_detailed_results(results, feature_results, best_metrics, selected_features,
                          feature_importance, best_params, train_val_metrics, filepath):
    try:
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            df_metrics = pd.DataFrame({
                'Dataset': ['Train', 'Test'],
                'MAE': [results['train']['MAE'], results['test']['MAE']],
                'R2': [results['train']['R2'], results['test']['R2']],
                'RMSE': [results['train']['RMSE'], results['test']['RMSE']],
                'Objetivos_Cumplidos': [results['train']['objectives_met'],
                                        results['test']['objectives_met']]
            })
            df_metrics.to_excel(writer, sheet_name='Métricas', index=False)

            peak_data = []
            for ds in ['train', 'test']:
                for p in [90, 95, 97.5]:
                    k = f'P{p}_MAE'
                    if k in results[ds]['peak_metrics']:
                        peak_data.append({
                            'Dataset': ds.capitalize(), 'Percentil': p,
                            'MAE_Pico': results[ds]['peak_metrics'][f'P{p}_MAE'],
                            'R2_Pico': results[ds]['peak_metrics'][f'P{p}_R2'],
                            'Threshold': results[ds]['peak_metrics'][f'P{p}_Threshold'],
                            'N_Picos': results[ds]['peak_metrics'][f'P{p}_Count']
                        })
            if peak_data:
                pd.DataFrame(peak_data).to_excel(writer, sheet_name='Métricas_Picos', index=False)

            if selected_features:
                imp = feature_importance if len(feature_importance) == len(selected_features) \
                      else [0]*len(selected_features)
                pd.DataFrame({'Feature': selected_features, 'Importance': imp}) \
                    .sort_values('Importance', ascending=False) \
                    .to_excel(writer, sheet_name='Características_Seleccionadas', index=False)

            pd.DataFrame({
                'Métrica': ['MAE', 'R²', 'RMSE', 'MAE_Picos(P90)', 'Objetivos_Cumplidos'],
                'Valor': [best_metrics['mae'], best_metrics['r2'], best_metrics['rmse'],
                          best_metrics['peak_mae'], best_metrics['objectives_met']]
            }).to_excel(writer, sheet_name='Mejor_Modelo_Métricas', index=False)

            if best_params:
                pd.DataFrame({'Parámetro': list(best_params.keys()),
                              'Valor': list(best_params.values())}) \
                    .to_excel(writer, sheet_name='Mejores_Hiperparámetros', index=False)

            if train_val_metrics:
                pd.DataFrame({
                    'Métrica': ['MAE Train', 'MAE Test', 'R² Train', 'R² Test',
                                'MAE Picos Test', 'Brecha Sobreajuste', 'Objetivos'],
                    'Valor': [train_val_metrics['mae_train'], train_val_metrics['mae_test'],
                              train_val_metrics['r2_train'], train_val_metrics['r2_test'],
                              train_val_metrics['peak_mae_test'],
                              train_val_metrics['overfit_gap'],
                              train_val_metrics['objectives_met']]
                }).to_excel(writer, sheet_name='Validación_Hiperparámetros', index=False)

                if 'all_results' in train_val_metrics:
                    df_all = train_val_metrics['all_results'].copy()
                    df_all['params'] = df_all['params'].astype(str)
                    df_all.to_excel(writer, sheet_name='Todos_Resultados_Tuning', index=False)

            if feature_results:
                pd.DataFrame(feature_results).to_excel(writer, sheet_name='Búsqueda_Features', index=False)

            max_len = max(len(results['train']['actual']), len(results['test']['actual']))
            def pad(a): return np.pad(a, (0, max_len - len(a)), constant_values=np.nan)
            pd.DataFrame({
                'Train_Actual': pad(results['train']['actual']),
                'Train_Predicted': pad(results['train']['predictions']),
                'Test_Actual': pad(results['test']['actual']),
                'Test_Predicted': pad(results['test']['predictions'])
            }).to_excel(writer, sheet_name='Predicciones', index=False)

        print(f"✅ Resultados guardados en: {filepath}")
    except Exception as e:
        print(f"⚠️ Error al guardar Excel: {e}")


def generate_deployment_script(output_dir, base_path):
    """Genera script de despliegue para el modelo LightGBM meteorológico."""
    script_content = '''"""
Script de despliegue del modelo de predicción de casos de dengue
usando SOLO predictores meteorológicos con LightGBM.
Generado automáticamente - {fecha}
"""
import pandas as pd
import numpy as np
import pickle
import json
import os

class DengueMeteoDeploymentLGBM:
    METEOROLOGICAL_PREDICTORS = {meteo_list}

    def __init__(self, model_dir):
        self.model_dir = model_dir
        self.load_model()

    def load_model(self):
        with open(os.path.join(self.model_dir, 'scaler.pkl'), 'rb') as f:
            self.scaler = pickle.load(f)
        with open(os.path.join(self.model_dir, 'lgbm_model.pkl'), 'rb') as f:
            self.lgbm_model = pickle.load(f)
        with open(os.path.join(self.model_dir, 'model_config.json'), 'r') as f:
            cfg = json.load(f)
            self.selected_features = cfg['selected_features']
            self.X_augmented_columns = cfg['X_augmented_columns']
            self.best_params = cfg.get('best_params', {{}})
        print("✅ Modelo LightGBM (meteorológico) cargado")

    def _filter_meteo(self, X):
        allowed = [c for c in X.columns if c in self.METEOROLOGICAL_PREDICTORS]
        return X[allowed].copy()

    def _interactions(self, X):
        d = pd.DataFrame(index=X.index)
        for a, b, n in [('temp','hum_rel','temp_hum_rel'),('temp_max','hum_rel','temp_max_hum_rel'),
                        ('prec','temp','prec_temp'),('dias_lluvia','hum_rel','dias_lluvia_hum_rel'),
                        ('soi_lag_12','sst_lag_12','soi_sst_lag12'),('temp','soi_lag_12','temp_soi_lag12'),
                        ('prec','hum_rel','prec_hum_rel')]:
            if a in X.columns and b in X.columns:
                d[n] = X[a]*X[b]
        return d

    def _polys(self, X):
        d = pd.DataFrame(index=X.index)
        for v in ['temp','hum_rel','prec','temp_max','temp_min']:
            if v in X.columns:
                d[v+'_squared'] = X[v]**2
        return d

    def _lag_aggs(self, X):
        d = pd.DataFrame(index=X.index)
        for b in ['temp','temp_max','temp_min','hum_esp','hum_rel','prec','dias_lluvia']:
            cols = [c for c in X.columns if c.startswith(b+'_lag_')]
            if len(cols) >= 3:
                l = X[cols]
                d[b+'_lag_mean'] = l.mean(axis=1); d[b+'_lag_std'] = l.std(axis=1)
                d[b+'_lag_max'] = l.max(axis=1); d[b+'_lag_min'] = l.min(axis=1)
        return d

    def _rolling(self, X):
        d = pd.DataFrame(index=X.index)
        if 'semana_epi' in X.columns:
            d['week_sin'] = np.sin(2*np.pi*X['semana_epi']/52)
            d['week_cos'] = np.cos(2*np.pi*X['semana_epi']/52)
        if 'temp' in X.columns and 'temp_lag_4' in X.columns:
            d['temp_trend_4w'] = X['temp'] - X['temp_lag_4']
        if 'temp' in X.columns and 'temp_lag_12' in X.columns:
            d['temp_trend_12w'] = X['temp'] - X['temp_lag_12']
        if 'prec' in X.columns and 'prec_lag_4' in X.columns:
            d['prec_trend_4w'] = X['prec'] - X['prec_lag_4']
        if 'hum_rel' in X.columns and 'hum_rel_lag_4' in X.columns:
            d['hum_trend_4w'] = X['hum_rel'] - X['hum_rel_lag_4']
        return d

    def _augment(self, X):
        Xm = self._filter_meteo(X)
        num = [c for c in Xm.columns
               if pd.api.types.is_numeric_dtype(Xm[c]) and c.lower() not in
               ['fecha','date','datetime','timestamp']]
        Xn = Xm[num]
        parts = [Xn]
        for fn in [self._interactions, self._polys, self._lag_aggs, self._rolling]:
            f = fn(Xn)
            if not f.empty: parts.append(f)
        Xa = pd.concat(parts, axis=1)
        Xa = Xa.replace([np.inf, -np.inf], np.nan).fillna(Xa.mean())
        return Xa

    def predict(self, X):
        Xa = self._augment(X)
        for c in self.X_augmented_columns:
            if c not in Xa.columns: Xa[c] = 0
        Xa = Xa[self.X_augmented_columns]
        Xs = pd.DataFrame(self.scaler.transform(Xa),
                          columns=Xa.columns, index=Xa.index)
        return self.lgbm_model.predict(Xs[self.selected_features])

if __name__ == "__main__":
    model_dir = r"{model_dir}"
    model = DengueMeteoDeploymentLGBM(model_dir)
    test_path = r"{test_path}"
    df = pd.read_excel(test_path)
    X_test = df.drop('casos_dengue', axis=1)
    preds = model.predict(X_test)
    print(f"Predicciones: {len(preds)}")
    print(f"  Media: {np.mean(preds):.2f} | Mediana: {np.median(preds):.2f}")
    out = os.path.join(model_dir, "predicciones_ejemplo.xlsx")
    res = X_test.copy(); res['prediccion_casos_dengue'] = preds
    res.to_excel(out, index=False)
    print(f"✅ Guardado en: {out}")
'''

    script_content = script_content.replace(
        '{fecha}', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    script_content = script_content.replace(
        '{meteo_list}', repr(DengueMeteoFeatureEngineeringLGBM.METEOROLOGICAL_PREDICTORS))

    script_path = os.path.join(output_dir, "modelo_despliegue_lightgbm_meteo.py")
    model_dir_abs = os.path.join(output_dir, "modelo_guardado_lightgbm_meteo")
    test_path_abs = os.path.join(
        base_path, "2_meteo_epi_2021-2026_1_rezagos_meteo_epi_test_90_10.xlsx")

    script_content = script_content.replace('{model_dir}', model_dir_abs)
    script_content = script_content.replace('{test_path}', test_path_abs)

    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(script_content)

    print(f"✅ Script de despliegue generado: {script_path}")
    return script_path


# ===================== EJECUCIÓN PRINCIPAL =====================

if __name__ == "__main__":
    print("="*60)
    print("LIGHTGBM - SOLO PREDICTORES METEOROLÓGICOS")
    print(f"OBJETIVOS: MAE < {MAE_TARGET} | MAE Picos < {PEAK_MAE_TARGET} | R² > {R2_TARGET}")
    print("="*60)

    base_path = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_LightGBM\2_datos\1_raw\1_90_10"
    output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_LightGBM\2_datos\1_raw\1_90_10\1_resultados_lightgbm_90_10_meteo"
    os.makedirs(output_dir, exist_ok=True)

    # ---- Cargar datos ----
    train_path = os.path.join(base_path, "2_meteo_epi_2021-2026_1_rezagos_meteo_epi_train_90_10.xlsx")
    print(f"\nCargando train: {train_path}")
    train_data = pd.read_excel(train_path)
    X_train = train_data.drop('casos_dengue', axis=1)
    y_train = train_data['casos_dengue']
    print(f"Train: {X_train.shape[0]} muestras, {X_train.shape[1]} columnas")

    # ---- Ingeniería de atributos (solo meteorológicos) ----
    engine = DengueMeteoFeatureEngineeringLGBM()
    X_scaled, X_augmented = engine.fit_transform(X_train, y_train)

    # ---- Búsqueda del número óptimo de características ----
    feature_range = range(10, min(70, X_scaled.shape[1]), 5)
    best_n_features, feature_results, best_metrics = engine.find_optimal_features(
        X_scaled, y_train, feature_range)

    # ---- Split para tuning ----
    X_train_tune, X_test_tune, y_train_tune, y_test_tune = train_test_split(
        X_scaled, y_train, test_size=0.2, random_state=42)
    X_train_selected = engine.transform_with_selected_features(X_train_tune)
    X_test_selected = engine.transform_with_selected_features(X_test_tune)

    # ---- Tuning de hiperparámetros ----
    best_params, model = engine.tune_hyperparameters(
        X_train_selected, y_train_tune, X_test_selected, y_test_tune)

    # ---- Evaluación en train ----
    print("\n" + "="*60)
    print("EVALUACIÓN EN TRAIN (modelo final)")
    print("="*60)
    X_train_full = engine.transform_with_selected_features(X_scaled)
    train_metrics = engine.evaluate_model_detailed(X_train_full, y_train, "Train")

    # ---- Cargar y evaluar en test real ----
    test_path = os.path.join(base_path, "2_meteo_epi_2021-2026_1_rezagos_meteo_epi_test_90_10.xlsx")
    print(f"\nCargando test: {test_path}")
    test_data = pd.read_excel(test_path)
    X_test = test_data.drop('casos_dengue', axis=1)
    y_test = test_data['casos_dengue']
    print(f"Test: {X_test.shape[0]} muestras, {X_test.shape[1]} columnas")

    X_test_scaled = engine.transform_features(X_test)
    X_test_selected_final = engine.transform_with_selected_features(X_test_scaled)

    print("\n" + "="*60)
    print("EVALUACIÓN EN TEST (datos reales)")
    print("="*60)
    test_metrics = engine.evaluate_model_detailed(X_test_selected_final, y_test, "Test")

    # ---- Resumen final ----
    overfit_gap_final = test_metrics['MAE'] - train_metrics['MAE']

    print("\n" + "="*60)
    print("RESUMEN FINAL - LIGHTGBM (SOLO METEOROLÓGICOS)")
    print("="*60)
    print(f"Train - MAE: {train_metrics['MAE']:.4f} | R²: {train_metrics['R2']:.4f}")
    print(f"Test  - MAE: {test_metrics['MAE']:.4f} | R²: {test_metrics['R2']:.4f}")
    print(f"Train - MAE P90: {train_metrics['peak_metrics'].get('P90_MAE', 0):.4f}")
    print(f"Test  - MAE P90: {test_metrics['peak_metrics'].get('P90_MAE', 0):.4f}")
    print(f"Brecha sobreajuste: {overfit_gap_final:.4f}")

    print("\n" + "-"*60)
    print("VERIFICACIÓN DE OBJETIVOS:")
    print("-"*60)
    mae_ok_train = train_metrics['MAE'] < MAE_TARGET
    mae_ok_test = test_metrics['MAE'] < MAE_TARGET
    peak_ok_train = train_metrics['peak_metrics'].get('P90_MAE', float('inf')) < PEAK_MAE_TARGET
    peak_ok_test = test_metrics['peak_metrics'].get('P90_MAE', float('inf')) < PEAK_MAE_TARGET
    r2_ok_train = train_metrics['R2'] > R2_TARGET
    r2_ok_test = test_metrics['R2'] > R2_TARGET

    print(f"  MAE < {MAE_TARGET}    -> Train: {'✅' if mae_ok_train else '❌'} "
          f"({train_metrics['MAE']:.4f}) | Test: {'✅' if mae_ok_test else '❌'} "
          f"({test_metrics['MAE']:.4f})")
    print(f"  MAE picos < {PEAK_MAE_TARGET} -> Train: {'✅' if peak_ok_train else '❌'} "
          f"({train_metrics['peak_metrics'].get('P90_MAE', 0):.4f}) | "
          f"Test: {'✅' if peak_ok_test else '❌'} "
          f"({test_metrics['peak_metrics'].get('P90_MAE', 0):.4f})")
    print(f"  R² > {R2_TARGET}   -> Train: {'✅' if r2_ok_train else '❌'} "
          f"({train_metrics['R2']:.4f}) | Test: {'✅' if r2_ok_test else '❌'} "
          f"({test_metrics['R2']:.4f})")

    all_ok = (mae_ok_train and mae_ok_test and peak_ok_train and
              peak_ok_test and r2_ok_train and r2_ok_test)
    print("\n" + "="*60)
    if all_ok:
        print("🎯 ¡TODOS LOS OBJETIVOS CUMPLIDOS EN TRAIN Y TEST!")
    else:
        print("⚠️ No se cumplieron todos los objetivos. Revisar resultados.")
    print("="*60)

    # ---- Guardar resultados ----
    results = {'train': train_metrics, 'test': test_metrics}
    excel_output = os.path.join(output_dir, "resultados_lightgbm_meteo.xlsx")
    save_detailed_results(
        results, feature_results, best_metrics,
        engine.selected_features,
        engine.lgbm_model.feature_importances_ if engine.lgbm_model is not None else [],
        best_params, engine.train_val_metrics, excel_output)

    plot_feature_selection_results(
        feature_results, best_metrics,
        save_path=os.path.join(output_dir, "feature_selection_analysis_lightgbm_meteo.png"))

    if engine.train_val_metrics and 'all_results' in engine.train_val_metrics:
        plot_hyperparameter_tuning_results(
            engine.train_val_metrics['all_results'],
            save_path=os.path.join(output_dir, "hyperparameter_tuning_results_lightgbm_meteo.png"))

    plot_peak_analysis(
        y_train, y_test, train_metrics, test_metrics,
        save_path=os.path.join(output_dir, "peak_analysis_lightgbm_meteo.png"))

    plot_performance_comparison(
        train_metrics, test_metrics,
        save_path=os.path.join(output_dir, "performance_comparison_lightgbm_meteo.png"))

    # ---- Guardar modelo ----
    model_dir = os.path.join(output_dir, "modelo_guardado_lightgbm_meteo")
    os.makedirs(model_dir, exist_ok=True)

    with open(os.path.join(model_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(engine.scaler, f)
    with open(os.path.join(model_dir, 'lgbm_model.pkl'), 'wb') as f:
        pickle.dump(engine.lgbm_model, f)

    config = {
        'predictors_used': 'SOLO METEOROLÓGICOS',
        'meteorological_predictors': engine.METEOROLOGICAL_PREDICTORS,
        'selected_features': engine.selected_features,
        'X_augmented_columns': engine.X_augmented_columns,
        'numeric_columns': engine.numeric_columns,
        'feature_names': engine.feature_names,
        'best_params': best_params,
        'best_metrics': best_metrics,
        'objectives': {
            'MAE_TARGET': MAE_TARGET, 'PEAK_MAE_TARGET': PEAK_MAE_TARGET,
            'R2_TARGET': R2_TARGET
        },
        'train_mae': train_metrics['MAE'], 'test_mae': test_metrics['MAE'],
        'train_r2': train_metrics['R2'], 'test_r2': test_metrics['R2'],
        'train_peak_mae': train_metrics['peak_metrics'].get('P90_MAE', 0),
        'test_peak_mae': test_metrics['peak_metrics'].get('P90_MAE', 0),
        'overfit_gap': overfit_gap_final,
        'all_objectives_met': bool(all_ok)
    }
    with open(os.path.join(model_dir, 'model_config.json'), 'w') as f:
        json.dump(config, f, indent=4)

    generate_deployment_script(output_dir, base_path)

    print("\n" + "="*60)
    print("PROCESO COMPLETADO")
    print("="*60)
    print(f"Resultados: {excel_output}")
    print(f"Modelo: {model_dir}")
    print(f"Script de despliegue: {os.path.join(output_dir, 'modelo_despliegue_lightgbm_meteo.py')}")